# Notebook 04 — Team Persona Engine

**Purpose:** Students don't work alone in real companies. This pipeline generates realistic Slack-style messages from AI teammates — a team lead, a senior developer, a client, and an HR person. These messages add texture and realism to the simulation.

**Personas generated:**
| Persona | Communication Style | When They Message |
|---|---|---|
| Priya (Team Lead) | Professional, direct, uses bullet points | Task assignments, check-ins, deadline reminders |
| Karan (Senior Dev) | Casual, helpful, uses slang | Code tips, "hey have you tried..." |
| Client (varies) | Formal email style, sometimes urgent | Requirement changes, feedback on deliverables |
| Sneha (HR) | Warm, encouraging | Welcome messages, performance updates |

**AI Model:** Groq — `llama-3.3-70b-versatile` via `langchain-groq`

**Key Techniques:**
- Strong persona definitions (name, age, years of experience, communication quirks)
- Persona memory: each persona "remembers" their previous messages to the student
- Context injection: the message must reference the actual task the student just submitted
- Temperature tuning: slightly higher temperature for more natural, varied messages

**Input:** `{ persona, trigger, context, student_performance }`

**Output:** Structured JSON with persona name, role, message, message type, and timestamp offset

---
## 1. Setup & Dependencies

In [1]:
# %pip install -r requirements.txt

In [2]:
import os
import json
import random
from datetime import datetime, timedelta
from typing import TypedDict, Optional, List
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END

load_dotenv()

print("All imports loaded")
print(f"Groq API key present: {bool(os.getenv('GROQ_API_KEY'))}")

C:\Users\sankb\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports loaded
Groq API key present: True


C:\Users\sankb\miniconda3\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")

MODEL_NAME = "llama-3.3-70b-versatile"

model = ChatGroq(
    model=MODEL_NAME,
    temperature=0.8,  # slightly higher for natural, varied messages
    max_tokens=1024,
    api_key=GROQ_API_KEY,
)

print(f"ChatGroq initialized with model: {MODEL_NAME}")

ChatGroq initialized with model: llama-3.3-70b-versatile


---
## 2. Define Persona State (LangGraph)

In [4]:
class PersonaState(TypedDict):
    persona: str                # "team_lead" | "senior_dev" | "client" | "hr"
    trigger: str                # what just happened (e.g. "task_submitted")
    context: dict               # task + submission info
    student_performance: str    # "good" | "needs_improvement" | "excellent"
    persona_definition: Optional[dict]
    previous_messages: list     # past messages from this persona for memory
    result: Optional[dict]
    error: Optional[str]

print("PersonaState defined")

PersonaState defined


---
## 3. Persona Definitions & System Prompts

In [5]:
PERSONAS = {
    "team_lead": {
        "name": "Priya",
        "role": "Team Lead",
        "age": 32,
        "years_experience": 9,
        "communication_style": "Professional, direct, uses bullet points. Gives clear instructions and constructive feedback. Occasionally uses emoji like \U0001f4aa or \U0001f44d.",
        "personality": "Organized, mentoring, slightly strict about deadlines but always fair. She cares about team growth and code quality.",
        "message_type": "slack",
        "sign_off": "",
    },
    "senior_dev": {
        "name": "Karan",
        "role": "Senior Developer",
        "age": 29,
        "years_experience": 7,
        "communication_style": "Casual, friendly, uses slang like 'hey', 'gotta', 'nice one'. Drops informal tips and references pop culture. Very approachable.",
        "personality": "Laid-back but brilliant. Loves mentoring juniors, cracks jokes, but gives sharp technical feedback when needed.",
        "message_type": "slack",
        "sign_off": "",
    },
    "client": {
        "name": "Rohan",
        "role": "Client",
        "age": 40,
        "years_experience": 18,
        "communication_style": "Formal, polite, business-oriented. Writes in complete paragraphs. Sometimes uses urgent language. Uses email format with Subject line.",
        "personality": "Busy executive who values clear communication and timely delivery. Appreciates thoroughness but gets impatient with delays.",
        "message_type": "email",
        "sign_off": "\n\nBest regards,\nRohan\nVP of Engineering, TechCorp",
    },
    "hr": {
        "name": "Sneha",
        "role": "HR Manager",
        "age": 35,
        "years_experience": 11,
        "communication_style": "Warm, encouraging, supportive. Uses positive language and emoji like \U0001f60a or \U0001f44f. Writes in a friendly, conversational tone.",
        "personality": "Empathetic and people-focused. She checks in on wellbeing, celebrates wins, and offers support during stressful periods.",
        "message_type": "slack",
        "sign_off": "",
    },
}

print(f"Loaded {len(PERSONAS)} persona definitions")

Loaded 4 persona definitions


In [6]:
TRIGGER_TEMPLATES = {
    "task_submitted": [
        "The student has just submitted their completed task. Generate a message reacting to their submission. Reference the specific task title and their work.",
        "triggers: task submission reaction, feedback on deliverable"
    ],
    "deadline_approaching": [
        "The deadline for the current task is approaching soon. Generate a reminder or check-in message. Reference the deadline and encourage progress.",
        "triggers: deadline reminder, urgency"
    ],
    "task_assigned": [
        "A new task has been assigned to the student. Generate a message providing context and expectations. Explain the task and why it matters.",
        "triggers: task assignment, instructions"
    ],
    "welcome": [
        "The student has just joined the team. Generate a welcome message introducing yourself and setting expectations. Make them feel part of the team.",
        "triggers: onboarding, first day"
    ],
    "performance_update": [
        "The student has completed a significant milestone. Generate a message about their performance so far. Reference their specific achievements and areas for growth.",
        "triggers: progress review, feedback"
    ],
}

print(f"Loaded {len(TRIGGER_TEMPLATES)} trigger templates")

Loaded 5 trigger templates


---
## 4. Build LangGraph Persona Nodes

In [7]:
def load_persona_node(state: PersonaState) -> dict:
    """Select the persona definition based on persona key."""
    persona_key = state["persona"]
    definition = PERSONAS.get(persona_key)

    if not definition:
        return {"error": f"Unknown persona: {persona_key}"}

    print(f"  [load_persona] Loaded: {definition['name']} ({definition['role']})")
    return {"persona_definition": definition}

print("load_persona_node defined")

load_persona_node defined


In [8]:
def generate_message_node(state: PersonaState) -> dict:
    """Generate a persona message using the persona definition and context."""
    definition = state["persona_definition"]
    persona_key = state["persona"]
    trigger = state.get("trigger", "task_submitted")
    context = state.get("context", {})
    performance = state.get("student_performance", "good")
    previous = state.get("previous_messages", [])

    trigger_info = TRIGGER_TEMPLATES.get(trigger, TRIGGER_TEMPLATES["task_submitted"])

    # Format previous messages for memory
    memory_lines = []
    if previous:
        memory_lines.append("## Previous Messages from You (Memory)")
        for i, msg in enumerate(previous[-3:], 1):  # last 3 messages
            memory_lines.append(f"{i}. \"{msg}\"")
        memory_lines.append("")

    # Format context
    context_json = json.dumps(context, indent=2) if context else "{}"

    # Performance modifier
    performance_notes = {
        "excellent": "The student has been performing excellently. Be very positive and praise their work specifically.",
        "good": "The student has been performing well. Acknowledge their effort and give constructive feedback.",
        "needs_improvement": "The student needs improvement. Be encouraging but direct about what needs to change.",
    }
    perf_note = performance_notes.get(performance, performance_notes["good"])

    # Build system prompt
    system_lines = [
        "You are an AI that generates realistic workplace messages from a specific team member.",
        "",
        "## Your Identity",
        f"Name: {definition["name"]}",
        f"Role: {definition["role"]}",
        f"Age: {definition["age"]}",
        f"Years of Experience: {definition["years_experience"]}",
        "",
        "## Your Communication Style",
        f"{definition["communication_style"]}",
        "",
        "## Your Personality",
        f"{definition["personality"]}",
        "",
        "## Performance Context",
        f"{perf_note}",
        "",
    ]
    system_lines.extend(memory_lines)
    system_lines.extend([
        "## Trigger for This Message",
        f"Trigger: {trigger}",
        f"{trigger_info[0]}",
        "",
        "## Task Context",
        f"{context_json}",
        "",
        "## Output Requirements",
        "Respond with a valid JSON object containing:",
        "  - message: string (the actual message content, use \\n for line breaks)",
        f"  - message_type: \"{definition["message_type"]}\" (slack or email)",
        "  - timestamp_offset_minutes: int (minutes after the event, 1-120)",
        "",
        "IMPORTANT: Stay in character 100%. Write exactly how this person would write. Reference the specific task details.",
    ])

    system_prompt = "\n".join(system_lines)

    try:
        response = model.bind(response_format={"type": "json_object"}).invoke([
            SystemMessage(content=system_prompt),
        ])
        content = response.content

        result = json.loads(content)

        required = ["message", "message_type", "timestamp_offset_minutes"]
        for key in required:
            if key not in result:
                raise ValueError(f"Response missing required field: {key}")

        # Enrich result with persona metadata
        result["persona_name"] = definition["name"]
        result["persona_role"] = definition["role"]
        if definition.get("sign_off"):
            result["message"] += definition["sign_off"]

        print(f"  [generate_message] {definition["name"]} ({definition["role"]})")
        print(f"    Trigger: {trigger}, Performance: {performance}")
        print(f"    Type: {result["message_type"]}, Offset: {result["timestamp_offset_minutes"]}min")

        return {"result": result}
    except Exception as e:
        print(f"  [generate_message] Failed: {e}")
        return {"error": str(e)}

print("generate_message_node defined")

generate_message_node defined


---
## 5. Compile the LangGraph

In [9]:
def build_persona_graph():
    workflow = StateGraph(PersonaState)

    workflow.add_node("load_persona", load_persona_node)
    workflow.add_node("generate_message", generate_message_node)

    workflow.set_entry_point("load_persona")
    workflow.add_edge("load_persona", "generate_message")
    workflow.add_edge("generate_message", END)

    return workflow.compile()

app = build_persona_graph()
print("LangGraph Team Persona Engine compiled successfully")

LangGraph Team Persona Engine compiled successfully


---
## 6. Helper Functions

In [10]:
def generate_persona_message(
    persona: str,
    trigger: str = "task_submitted",
    context: dict = None,
    student_performance: str = "good",
    previous_messages: list = None,
) -> dict:
    """Generate a message from a specific persona."""
    state = PersonaState(
        persona=persona,
        trigger=trigger,
        context=context or {},
        student_performance=student_performance,
        persona_definition=None,
        previous_messages=previous_messages or [],
        result=None,
        error=None,
    )

    try:
        final = app.invoke(state)
        return final
    except Exception as e:
        print(f"Failed to generate message: {e}")
        return {"error": str(e)}

print("generate_persona_message function defined")

generate_persona_message function defined


In [11]:
def display_message(state: dict):
    if state.get("error"):
        print(f"[ERROR] {state["error"]}")
        return

    result = state.get("result", {})
    print("=" * 70)
    print("TEAM PERSONA MESSAGE")
    print("=" * 70)
    print(f"From: {result.get("persona_name", "?")} ({result.get("persona_role", "?")})")
    print(f"Type: {result.get("message_type", "?").upper()}")
    print(f"Offset: {result.get("timestamp_offset_minutes", "?")}min after event")
    print("-" * 70)
    print(result.get("message", "N/A"))
    print("-" * 70)

print("display_message function defined")

display_message function defined


In [12]:
def bulk_generate_messages(
    persona: str,
    count: int = 5,
    context: dict = None,
    trigger: str = "task_submitted",
    performance: str = "good",
) -> list:
    """Generate multiple messages from the same persona to check consistency."""
    results = []
    for i in range(count):
        state = generate_persona_message(
            persona=persona,
            trigger=trigger,
            context=context,
            student_performance=performance,
        )
        if state.get("error"):
            print(f"  [{i+1}] FAILED: {state["error"]}")
        else:
            msg = state["result"].get("message", "")
            results.append(state)
            preview = msg[:80].replace("\n", " ") + "..." if len(msg) > 80 else msg
            print(f"  [{i+1}] {state["result"].get("persona_name", "?")}: {preview}")

    print(f"\nGenerated {len(results)}/{count} messages from {persona}")
    return results

print("bulk_generate_messages function defined")

bulk_generate_messages function defined


---
## 7. Tests

### Test 1: Generate 5 messages from each persona — check consistency

In [13]:
print("=" * 70)
print("TEST 1: Generate 5 messages from each persona")
print("=" * 70)

sample_context = {
    "task_id": "task_001",
    "title": "Fix Authentication Token Expiry Bug",
    "description": "Users are being logged out prematurely. Fix the JWT token expiry handling.",
    "task_type": "bug_fix",
    "difficulty": "medium",
    "deadline_hours": 4,
    "submission_summary": "Implemented token refresh mechanism using axios interceptors with proper error handling and refresh token rotation."
}

print("\n--- Generating messages from ALL personas ---\n")

for persona_key in ["team_lead", "senior_dev", "client", "hr"]:
    p = PERSONAS[persona_key]
    print(f"\n>>> {p["name"]} ({p["role"]}) <<<")
    print("-" * 40)
    results = bulk_generate_messages(
        persona=persona_key,
        count=5,
        context=sample_context,
        trigger="task_submitted",
        performance="good",
    )
    print()

print("=" * 70)
print("TEST 1 COMPLETE")
print("=" * 70)

TEST 1: Generate 5 messages from each persona

--- Generating messages from ALL personas ---


>>> Priya (Team Lead) <<<
----------------------------------------
  [load_persona] Loaded: Priya (Team Lead)


  [generate_message] Priya (Team Lead)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 10min
  [1] Priya: Great work on submitting the 'Fix Authentication Token Expiry Bug' task 🙌!  I've...
  [load_persona] Loaded: Priya (Team Lead)


  [generate_message] Priya (Team Lead)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 5min
  [2] Priya: Hey, great work on submitting the 'Fix Authentication Token Expiry Bug' task 🙌! ...
  [load_persona] Loaded: Priya (Team Lead)


  [generate_message] Priya (Team Lead)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 5min
  [3] Priya: Great job on submitting the 'Fix Authentication Token Expiry Bug' task, 👍 Your i...
  [load_persona] Loaded: Priya (Team Lead)


  [generate_message] Priya (Team Lead)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 5min
  [4] Priya: Great job on completing the 'Fix Authentication Token Expiry Bug' task, 👍 Your i...
  [load_persona] Loaded: Priya (Team Lead)


...

  [generate_message] Sneha (HR Manager)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 5min
  [1] Sneha: Hey there, 👏 I just wanted to reach out and say great job on completing the 'Fix...
  [load_persona] Loaded: Sneha (HR Manager)


  [generate_message] Sneha (HR Manager)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 5min
  [2] Sneha: 👏 Great job on submitting the 'Fix Authentication Token Expiry Bug' task! 😊 I ju...
  [load_persona] Loaded: Sneha (HR Manager)


  [generate_message] Sneha (HR Manager)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 5min
  [3] Sneha: Hey, I just saw that you submitted your 'Fix Authentication Token Expiry Bug' ta...
  [load_persona] Loaded: Sneha (HR Manager)


  [generate_message] Sneha (HR Manager)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 5min
  [4] Sneha: Hey 👏, just wanted to say great job on submitting the 'Fix Authentication Token ...
  [load_persona] Loaded: Sneha (HR Manager)


  [generate_message] Sneha (HR Manager)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 5min
  [5] Sneha: Hey there, 👏 I just wanted to reach out and say great job on submitting the 'Fix...

Generated 5/5 messages from hr

TEST 1 COMPLETE


### Test 2: Karan (casual) vs Client (formal) — verify tone difference

In [14]:
print("=" * 70)
print("TEST 2: Tone Comparison - Karan (casual) vs Client (formal)")
print("=" * 70)

tone_context = {
    "task_id": "task_002",
    "title": "Build User Dashboard UI",
    "description": "Create a responsive dashboard with charts, tables, and filters.",
    "task_type": "coding",
    "difficulty": "medium",
    "submission_summary": "Built dashboard with React, Recharts, and Tailwind CSS. Includes date range filtering and export to CSV.",
}

print("\n--- KARAN (Senior Dev - Casual) ---")
karan_result = generate_persona_message(
    persona="senior_dev",
    trigger="task_submitted",
    context=tone_context,
    student_performance="good",
)
display_message(karan_result)

print("\n--- CLIENT (Formal) ---")
client_result = generate_persona_message(
    persona="client",
    trigger="task_submitted",
    context=tone_context,
    student_performance="good",
)
display_message(client_result)

if karan_result.get("result") and client_result.get("result"):
    karan_msg = karan_result["result"].get("message", "")
    client_msg = client_result["result"].get("message", "")
    # Check for casual markers in Karan vs formal markers in Client
    karan_casual_words = ["hey", "nice", "awesome", "gotta", "dude", "btw", "lmk"]
    client_formal_words = ["regards", "sincerely", "appreciate", "kindly", "please", "formal", "subject"]
    karan_score = sum(1 for w in karan_casual_words if w in karan_msg.lower())
    client_score = sum(1 for w in client_formal_words if w in client_msg.lower())
    print(f"\n--- Tone Analysis ---")
    print(f"Karan casual markers: {karan_score} ({karan_casual_words})")
    print(f"Client formal markers: {client_score} ({client_formal_words})")
    if karan_score >= 1 or client_score >= 1:
        print("PASS: Tone difference detected between Karan and Client")
    else:
        print("WARNING: Tone markers not clearly detected, but styles may still differ")
else:
    print("WARNING: One or both messages failed to generate")

TEST 2: Tone Comparison - Karan (casual) vs Client (formal)

--- KARAN (Senior Dev - Casual) ---
  [load_persona] Loaded: Karan (Senior Developer)


  [generate_message] Karan (Senior Developer)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 5min
TEAM PERSONA MESSAGE
From: Karan (Senior Developer)
Type: SLACK
Offset: 5min after event
----------------------------------------------------------------------
Hey, nice one on completing the Build User Dashboard UI task! 
I've taken a look at your submission and I gotta say, I'm impressed with how you've used React, Recharts, and Tailwind CSS to create a responsive dashboard. The date range filtering and export to CSV features are really cool. 
One thing to consider for future tasks is adding more comments to your code, especially for complex components. This will make it easier for others to understand your thought process and implement your code. 
Overall, great job and keep up the good work!
----------------------------------------------------------------------

--- CLIENT (Formal) ---
  [load_persona] Loaded: Rohan (Client)


  [generate_message] Rohan (Client)
    Trigger: task_submitted, Performance: good
    Type: email, Offset: 30min
TEAM PERSONA MESSAGE
From: Rohan (Client)
Type: EMAIL
Offset: 30min after event
----------------------------------------------------------------------
Subject: Re: Task Submission - Build User Dashboard UI

Dear Team,

I have reviewed the submitted task, Build User Dashboard UI, and I must commend the effort put into creating a responsive dashboard with the required features. The use of React, Recharts, and Tailwind CSS is impressive, and the inclusion of date range filtering and export to CSV functionality is a great addition.

However, I would like to request a minor revision to improve the overall user experience. Could you please enhance the dashboard's loading animation to make it more seamless and engaging?

Overall, the work is of high quality, and I appreciate the timely submission. Please let me know if you have any questions or concerns regarding the feedback.

Be

### Test 3: Bad submission vs Good submission — check tone adjusts

In [15]:
print("=" * 70)
print("TEST 3: Performance-based tone adjustment")
print("=" * 70)

task_context = {
    "task_id": "task_003",
    "title": "Implement Payment Checkout Flow",
    "description": "Build a complete payment checkout with Stripe integration.",
    "task_type": "coding",
    "difficulty": "hard",
    "deadline_hours": 8,
    "submission_summary": "Student submission details here.",
}

print("\n--- GOOD PERFORMANCE ---")
good = generate_persona_message(
    persona="team_lead",
    trigger="task_submitted",
    context=task_context,
    student_performance="excellent",
)
display_message(good)

print("\n--- NEEDS IMPROVEMENT ---")
bad = generate_persona_message(
    persona="team_lead",
    trigger="task_submitted",
    context=task_context,
    student_performance="needs_improvement",
)
display_message(bad)

if good.get("result") and bad.get("result"):
    good_msg = good["result"].get("message", "")
    bad_msg = bad["result"].get("message", "")
    good_positive = sum(1 for w in ["great", "nice", "awesome", "excellent", "good job", "impressive"] if w in good_msg.lower())
    bad_critical = sum(1 for w in ["needs", "improve", "fix", "issue", "problem", "concern", "but", "however"] if w in bad_msg.lower())
    print(f"\n--- Tone Comparison ---")
    print(f"Good message positive markers: {good_positive}")
    print(f"Bad message critical markers: {bad_critical}")
    if good_positive >= 1 and bad_critical >= 1:
        print("PASS: Tone differs based on performance (good=positive, bad=critical)")
    elif good_positive >= 1:
        print("PASS: Good performance received positive feedback")
    else:
        print("WARNING: Tone differentiation could be more pronounced")
else:
    print("WARNING: One or both messages failed to generate")

TEST 3: Performance-based tone adjustment

--- GOOD PERFORMANCE ---
  [load_persona] Loaded: Priya (Team Lead)


  [generate_message] Priya (Team Lead)
    Trigger: task_submitted, Performance: excellent
    Type: slack, Offset: 5min
TEAM PERSONA MESSAGE
From: Priya (Team Lead)
Type: SLACK
Offset: 5min after event
----------------------------------------------------------------------
Great work on submitting the 'Implement Payment Checkout Flow' task! 🚀
I've reviewed your code and I must say, the Stripe integration is seamless. Your attention to detail and adherence to best practices are truly commendable. Here are some highlights:
* Clean and readable code
* Well-structured payment flow
* Excellent error handling
Keep up the fantastic work! 💪 Your dedication to delivering high-quality solutions is appreciated. Looking forward to seeing your next task submission. 👍
----------------------------------------------------------------------

--- NEEDS IMPROVEMENT ---
  [load_persona] Loaded: Priya (Team Lead)


  [generate_message] Priya (Team Lead)
    Trigger: task_submitted, Performance: needs_improvement
    Type: slack, Offset: 30min
TEAM PERSONA MESSAGE
From: Priya (Team Lead)
Type: SLACK
Offset: 30min after event
----------------------------------------------------------------------
Hey, I've reviewed your submission for the 'Implement Payment Checkout Flow' task. Here are my thoughts:
* I like how you've structured the Stripe integration, it's clean and easy to follow.
* However, there are a few areas that need improvement: 
  * Error handling can be more robust.
  * Some functions can be optimized for better performance.
Keep working on it, you're getting closer! 💪
Let's discuss the details in our next meeting. Looking forward to seeing the updates.
----------------------------------------------------------------------

--- Tone Comparison ---
Good message positive markers: 2
Bad message critical markers: 2
PASS: Tone differs based on performance (good=positive, bad=critical)


### Test 4: Verify messages reference actual task details (not generic)

In [16]:
print("=" * 70)
print("TEST 4: Task Detail Reference Check")
print("=" * 70)

detail_context = {
    "task_id": "task_004",
    "title": "Dockerize Microservices Architecture",
    "description": "Create Dockerfiles and docker-compose for a microservices setup with 3 services.",
    "task_type": "devops",
    "difficulty": "hard",
    "submission_summary": "Created Dockerfiles for Node.js services, PostgreSQL, and Redis. Set up docker-compose with networking and volumes.",
}

keywords_to_check = ["docker", "microservice", "docker-compose", "container", "service"]

for persona_key in ["team_lead", "senior_dev", "client", "hr"]:
    p = PERSONAS[persona_key]
    print(f"\n--- {p["name"]} ({p["role"]}) ---")
    result = generate_persona_message(
        persona=persona_key,
        trigger="task_submitted",
        context=detail_context,
        student_performance="good",
    )

    if result.get("result"):
        msg = result["result"].get("message", "").lower()
        matched = [kw for kw in keywords_to_check if kw in msg]
        print(f"  Keywords found: {matched if matched else 'NONE'}")
        if matched:
            print(f"  PASS: Message references task details ({len(matched)} keywords)")
        else:
            print(f"  WARNING: Message may be generic (no task-specific keywords)")
    else:
        print(f"  FAIL: Message generation failed")

print("\n" + "=" * 70)
print("TEST 4 COMPLETE")

TEST 4: Task Detail Reference Check

--- Priya (Team Lead) ---
  [load_persona] Loaded: Priya (Team Lead)


  [generate_message] Priya (Team Lead)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 5min
  Keywords found: ['docker', 'microservice', 'docker-compose', 'service']
  PASS: Message references task details (4 keywords)

--- Karan (Senior Developer) ---
  [load_persona] Loaded: Karan (Senior Developer)


  [generate_message] Karan (Senior Developer)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 5min
  Keywords found: ['docker', 'microservice', 'docker-compose', 'service']
  PASS: Message references task details (4 keywords)

--- Rohan (Client) ---
  [load_persona] Loaded: Rohan (Client)


  [generate_message] Rohan (Client)
    Trigger: task_submitted, Performance: good
    Type: email, Offset: 30min
  Keywords found: ['docker', 'microservice', 'docker-compose', 'service']
  PASS: Message references task details (4 keywords)

--- Sneha (HR Manager) ---
  [load_persona] Loaded: Sneha (HR Manager)


  [generate_message] Sneha (HR Manager)
    Trigger: task_submitted, Performance: good
    Type: slack, Offset: 5min
  Keywords found: ['docker', 'microservice', 'docker-compose', 'service']
  PASS: Message references task details (4 keywords)

TEST 4 COMPLETE


---
## 8. Summary

### Test Results Checklist
- [ ] **Test 1:** Generate 5 messages from each persona — check they sound like the same person each time
- [ ] **Test 2:** Test Karan (casual) and Client (formal) messages for the same event — verify tone difference
- [ ] **Test 3:** Test message after a bad submission vs a good one — check tone adjusts appropriately
- [ ] **Test 4:** Verify messages reference actual task details (not generic)

### Architecture
```
StateGraph(PersonaState)
  [load_persona] -> [generate_message] -> END
       |                  |
  Selects persona     Calls ChatGroq with
  definition          persona identity +
  from dict           context + trigger
```

### Personas
| Key | Name | Role | Style |
|-----|------|------|-------|
| team_lead | Priya | Team Lead | Professional, direct, bullet points |
| senior_dev | Karan | Senior Developer | Casual, helpful, slang |
| client | Rohan | Client | Formal, email style |
| hr | Sneha | HR Manager | Warm, encouraging |

### Performance-based tone mapping
| Performance | Tone |
|-------------|------|
| excellent | Very positive, praise specific work |
| good | Acknowledge effort, constructive feedback |
| needs_improvement | Encouraging but direct about changes needed |

### Output Schema
```python
{
  "persona_name": "Priya",
  "persona_role": "Team Lead",
  "message": "Hey! Great work on the auth fix...",
  "message_type": "slack",       # slack | email
  "timestamp_offset_minutes": 15
}
```